<a href="https://colab.research.google.com/github/Lagnajit09/100x_AI_ML/blob/main/CalibratedTypedDecisionDatasetBuilder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Calibrated-Typed-Decisions (v1) - Dataset Builder**


---



---



Builds an open dataset of typed decisions (choice / yes-no) where every row has:
  - the human gold label (from a public labelled dataset)
  - the teacher's raw option log-probs (one forward pass, logit-slicing)
  - a calibrated soft label (teacher log-probs / T, with T fit by log loss
    against gold labels on a validation split)

## [1] Install (once per Colab session)

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 34.4 MB/s eta 0:00:00


## [2] Google Drive (survives disconnects) + Hugging Face login (needs a WRITE token)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from huggingface_hub import notebook_login
notebook_login()

## [3] Config

In [ ]:
import os, json, random
import numpy as np
import torch

SMOKE = False          # True: tiny end-to-end test run. False: the real build.

SEED        = 0
HF_REPO     = "m-lagnajit/calibrated-typed-decisions"
TEACHER     = "Qwen/Qwen2.5-7B-Instruct"
OUT_DIR     = "/content/drive/MyDrive/ctd_smoke" if SMOKE else "/content/drive/MyDrive/ctd"
N_PER_SPLIT = ({"train": 16, "validation": 16, "test": 16} if SMOKE
               else {"train": 1000, "validation": 300, "test": 300})
NOUL_RATE        = 0.3     # fraction of rows turned into yes/no questions
MIN_OPTIONS      = 3
MAX_OPTIONS      = 8
BATCH_SIZE       = 8
MAX_STATE_CHARS  = 1200
MIN_LETTER_MASS  = 0.5     # drop rows where the teacher didn't really answer with a letter
MIN_FIT_ROWS     = 30      # fewer validation rows than this = too few to trust a fitted temperature
DATASET_LICENSE  = "cc-by-4.0"   # set only after checking it's compatible with EVERY kept source
os.makedirs(OUT_DIR, exist_ok=True)

SOURCES = [
    dict(name="clinc_oos", path="clinc/clinc_oos", config="plus",
         text="text", label="intent", train="train", eval="test",
         rename={"oos": "out of scope"},
         instructions="Which intent best describes this request to a virtual assistant?",
         noul="Is this request asking for: {label}?",
         credit="CLINC150 (Larson et al., 2019), CC-BY-3.0"),
    dict(name="go_emotions", path="google-research-datasets/go_emotions", config="simplified",
         text="text", label="labels", multi=True, train="train", eval="test",
         instructions="Which emotion does this comment express?",
         noul="Does this comment express {label}?",
         credit="GoEmotions (Demszky et al., 2020), Apache-2.0"),
    # The four below load from MTEB's plain-file copies. A copy can't change the original
    # license, so LICENSE_OVERRIDES (below) uses each ORIGINAL dataset's license.
    dict(name="banking77", path="mteb/banking77", config=None,
         text="text", label="label", label_name_col="label_text", train="train", eval="test",
         instructions="Which banking intent best describes this customer message?",
         noul="Is the customer's intent in this message: {label}?",
         credit="BANKING77 (Casanueva et al., 2020), PolyAI/banking77, CC-BY-4.0; loaded via mteb/banking77"),
    dict(name="massive_intent", path="mteb/amazon_massive_intent", config="en",
         text="text", label="label", train="train", eval="test",    # labels are words here
         instructions="Which intent best describes this request to a voice assistant?",
         noul="Is this request to a voice assistant asking for: {label}?",
         credit="MASSIVE (FitzGerald et al., 2022), English subset, CC-BY-4.0; "
                "loaded via mteb/amazon_massive_intent"),
    # Toxicity (mteb/toxic_conversations_50k, CC0) was scored but left out of v1: the teacher
    # was at chance level (45% on two-option questions), so its soft labels carried no signal.
    dict(name="amazon_polarity", path="mteb/amazon_polarity", config=None,
         text="text", label="label", label_name_col="label_text", train="train", eval="test",
         instructions="What is the sentiment of this product review?",
         noul="Is the sentiment of this product review {label}?",
         credit="Amazon Review Polarity (Zhang et al., 2015), Apache-2.0 per fancyzhx/amazon_polarity; "
                "loaded via mteb/amazon_polarity"),
    # Left out: PolyAI/banking77 (old loading script, replaced by the MTEB copy above),
    # fancyzhx/ag_news and dair-ai/emotion (no clear license), and mteb/amazon_counterfactual
    # (the copy says CC-BY-4.0, but the original is CC-BY-SA-4.0: share-alike).
]

# Only these licenses pass the gate automatically. NC (non-commercial) and SA (share-alike)
# licenses, and anything "unknown"/"other", are excluded unless you read the card yourself.
PERMISSIVE = {"cc-by-4.0", "cc-by-3.0", "cc-by-2.0", "apache-2.0", "mit", "cc0-1.0"}
LICENSE_OVERRIDES = {                        # copy path -> ORIGINAL dataset's license
    "mteb/banking77":               "cc-by-4.0",   # PolyAI/banking77
    "mteb/amazon_massive_intent":   "cc-by-4.0",   # alexa/massive
    "mteb/toxic_conversations_50k": "cc0-1.0",     # Civil Comments / Jigsaw
}

## [4] License gate

In [ ]:
from huggingface_hub import dataset_info

def hub_license(path):
    try:
        cd = dataset_info(path).card_data
        lic = getattr(cd, "license", None) if cd is not None else None
    except Exception as e:
        return f"error:{type(e).__name__}"
    if isinstance(lic, list):
        lic = ",".join(lic)
    return (lic or "none").lower()

kept = []
print(f"{'source':42s} {'license':20s} kept?")
for src in SOURCES:
    lic = LICENSE_OVERRIDES.get(src["path"], hub_license(src["path"]))
    src["license"] = lic
    ok = lic in PERMISSIVE
    print(f"{src['path']:42s} {lic:20s} {'YES' if ok else 'no - check card manually'}")
    if ok:
        kept.append(src)
SOURCES = kept

source                                     license              kept?
clinc/clinc_oos                            cc-by-3.0            YES
google-research-datasets/go_emotions       apache-2.0           YES
mteb/banking77                             cc-by-4.0            YES
mteb/amazon_massive_intent                 cc-by-4.0            YES
mteb/amazon_polarity                       apache-2.0           YES


## [5] Load sources and build the remixed questions (no model needed yet)

In [ ]:
from datasets import load_dataset

def humanize(s):
    return s.replace("_", " ").strip()

def hub_load(src, split):
    """Load one split. Some older datasets (like Banking77) were published with a Python
    loading script, which recent versions of `datasets` refuse to run. For those, the Hub
    keeps an automatic Parquet copy on a branch called refs/convert/parquet, so we use that."""
    try:
        return load_dataset(src["path"], src["config"], split=split)
    except RuntimeError as e:
        if "scripts are no longer supported" not in str(e):
            raise
        print(f"  {src['name']}: old script dataset, loading the Parquet copy instead")
        return load_dataset(src["path"], src["config"] or "default", split=split,
                            revision="refs/convert/parquet")

def load_source(src):
    tr = hub_load(src, src["train"])
    ev = hub_load(src, src["eval"])
    feat = tr.features[src["label"]]
    src["_lab2id"] = None
    if src.get("multi"):
        names = feat.feature.names
    elif hasattr(feat, "names"):
        names = feat.names                      # the dataset has a built-in name list
    elif getattr(feat, "dtype", "") == "string":
        # Labels stored as words (e.g. MASSIVE's "alarm_set"): number them ourselves,
        # in alphabetical order so the numbering is the same on every run.
        names = sorted(set(tr.unique(src["label"])) | set(ev.unique(src["label"])))
        src["_lab2id"] = {n: i for i, n in enumerate(names)}
    else:
        # Labels stored as numbers, names in another column (e.g. MTEB's "label_text").
        # Read in chunks and stop once every label has been seen, so a 3.6-million-row
        # dataset doesn't have to be read in full.
        all_ids = {int(x) for ds in (tr, ev) for x in ds.unique(src["label"])}
        pairs = {}
        for ds in (tr, ev):
            for start in range(0, len(ds), 10_000):
                chunk = ds[start:start + 10_000]
                for lab, txt in zip(chunk[src["label"]], chunk[src["label_name_col"]]):
                    pairs.setdefault(int(lab), str(txt))
                if len(pairs) == len(all_ids):
                    break
            if len(pairs) == len(all_ids):
                break
        names = [pairs.get(i, f"label {i}") for i in range(max(all_ids) + 1)]
    rename = src.get("rename", {})
    return tr, ev, [rename.get(n, humanize(n)) for n in names]

def pick(ds, src, n, rng):
    """Pick n usable examples in a seeded random order."""
    order = list(range(len(ds)))
    rng.shuffle(order)
    out = []
    for i in order:
        ex = ds[i]
        lab = ex[src["label"]]
        if src.get("multi"):
            if len(lab) != 1:          # keep only single-label examples
                continue
            lab = lab[0]
        elif src.get("_lab2id"):
            lab = src["_lab2id"][lab]  # word label (e.g. "alarm_set") -> its number
        text = (ex[src["text"]] or "").strip()
        if not text:
            continue
        out.append((i, text[:MAX_STATE_CHARS], int(lab)))
        if len(out) == n:
            break
    return out

def make_question(src, names, gold, rng):
    """Turn one labelled example into a choice or yes/no question.
    display      = option texts in the order the teacher sees them
    display_perm = display position d shows canonical option display_perm[d]"""
    if rng.random() < NOUL_RATE:
        is_true = rng.random() < 0.5
        cand = gold if is_true else rng.choice([j for j in range(len(names)) if j != gold])
        if rng.random() < 0.5:
            display, perm = ["no", "yes"], [0, 1]     # canonical slots: [false, true]
        else:
            display, perm = ["yes", "no"], [1, 0]
        return dict(type="noul",
                    instructions=src["noul"].format(label=names[cand]),
                    options=["false", "true"], gold_index=int(is_true),
                    display=display, display_perm=perm)

    k = max(2, min(len(names), rng.randint(MIN_OPTIONS, MAX_OPTIONS)))
    others = rng.sample([j for j in range(len(names)) if j != gold], k - 1)
    opts = [gold] + others
    rng.shuffle(opts)                     # the right answer lands in a random slot
    options = [names[j] for j in opts]
    return dict(type="choice", instructions=src["instructions"],
                options=options, gold_index=opts.index(gold),
                display=options, display_perm=list(range(k)))

ALL = {"train": [], "validation": [], "test": []}
LOADED = []
for src in SOURCES:
    try:
        tr, ev, names = load_source(src)
    except Exception as e:
        print(f"SKIP {src['name']}: {type(e).__name__}: {e}")
        continue
    LOADED.append(src)
    rng = random.Random(f"{SEED}-{src['name']}")     # deterministic per source
    nv, nt = N_PER_SPLIT["validation"], N_PER_SPLIT["test"]
    picked = {"train": pick(tr, src, N_PER_SPLIT["train"], rng)}
    ev_rows = pick(ev, src, nv + nt, rng)            # val and test: disjoint halves of source test
    picked["validation"], picked["test"] = ev_rows[:nv], ev_rows[nv:]
    for split, items in picked.items():
        for (i, text, gold) in items:
            q = make_question(src, names, gold, rng)
            ALL[split].append(dict(
                id=f"{src['name']}-{split}-{i:06d}",
                source=src["path"], source_license=src["license"],
                state=text, gold=q["options"][q["gold_index"]], **q))
    print(f"{src['name']:12s} labels={len(names):3d}  " +
          "  ".join(f"{s}={len(v)}" for s, v in picked.items()))

print("rows:", {s: len(v) for s, v in ALL.items()})

clinc_oos    labels=151  train=1000  validation=300  test=300
go_emotions  labels= 28  train=1000  validation=300  test=300
banking77    labels= 77  train=1000  validation=300  test=300


Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

massive_intent labels= 60  train=1000  validation=300  test=300
amazon_polarity labels=  2  train=1000  validation=300  test=300
rows: {'train': 5000, 'validation': 1500, 'test': 1500}


## [6] Load the teacher in 4-bit (a 7B in fp16 needs ~15 GB, a T4 has ~15 GB total)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(TEACHER)
tok.padding_side = "left"        # so position -1 is the real last token for every row in a batch
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    TEACHER,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16),   # T4 has no bfloat16
    device_map="auto")
model.eval()

# Each answer letter can be tokenized with or without a leading space ("A" vs " A").
# After the chat template, the first answer token is usually "A" with no space.
# We collect both and combine them, instead of guessing one.
LETTER_IDS = []
for d in range(MAX_OPTIONS):
    L = chr(65 + d)
    ids = set()
    for s in (L, " " + L):
        enc = tok.encode(s, add_special_tokens=False)
        if len(enc) == 1:
            ids.add(enc[0])
    LETTER_IDS.append(sorted(ids))
print("letter token ids:", LETTER_IDS)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

letter token ids: [[32, 362], [33, 425], [34, 356], [35, 422], [36, 468], [37, 434], [38, 479], [39, 472]]


## [7] Score every row with ONE forward pass (batched, resumable)

In [ ]:
SYSTEM = "You are a careful text classifier. Reply with only the letter of the single best option."

def build_prompt(r):
    block = "\n".join(f"{chr(65 + d)}. {t}" for d, t in enumerate(r["display"]))
    user = (f"{r['instructions']}\n\nText:\n{r['state']}\n\n"
            f"Options:\n{block}\n\nReply with only the letter.")
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def score_batch(rows):
    enc = tok([build_prompt(r) for r in rows], return_tensors="pt", padding=True).to(model.device)
    try:
        out = model(**enc, logits_to_keep=1)   # only run the LM head at the last position
    except TypeError:
        out = model(**enc)                     # older transformers: head runs at every position
    logp = torch.log_softmax(out.logits[:, -1, :].float(), dim=-1)    # (B, vocab)
    results = []
    for b, r in enumerate(rows):
        n = len(r["display"])
        disp = torch.stack([torch.logsumexp(logp[b, LETTER_IDS[d]], dim=0)
                            for d in range(n)]).cpu()                 # display order
        canon = torch.empty_like(disp)
        canon[torch.tensor(r["display_perm"])] = disp                 # back to canonical order
        results.append(dict(
            teacher_option_logprobs=[round(x, 5) for x in canon.tolist()],
            teacher_letter_mass=round(disp.exp().sum().item(), 5)))
    return results

def read_valid(path):
    """Read a .jsonl file, skipping any half-written line left by a disconnect,
    then rewrite the file clean so new rows never get glued onto a broken line."""
    rows = []
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass                      # broken line: that row will simply be re-scored
        with open(path, "w") as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return rows

def run_split(split):
    path = f"{OUT_DIR}/{split}.jsonl"
    done = {r["id"] for r in read_valid(path)}
    todo = [r for r in ALL[split] if r["id"] not in done]
    print(f"{split}: {len(done)} already done, {len(todo)} to score")
    with open(path, "a") as f:
        for s in range(0, len(todo), BATCH_SIZE):
            batch = todo[s:s + BATCH_SIZE]
            for r, extra in zip(batch, score_batch(batch)):
                f.write(json.dumps({**r, **extra}, ensure_ascii=False) + "\n")
            f.flush()
            if (s // BATCH_SIZE) % 25 == 0:
                print(f"  {split}: {len(done) + s + len(batch)}/{len(ALL[split])}")
    print(f"  {split}: finished, {len(ALL[split])} rows")

# validation + test first: you can check calibration quality before spending time on train
for split in ["validation", "test", "train"]:
    run_split(split)

validation: 1800 already done, 0 to score
  validation: finished, 1500 rows
test: 1800 already done, 0 to score
  test: finished, 1500 rows
train: 6000 already done, 0 to score
  train: finished, 5000 rows


## [8] Fit one temperature per (source, type) on VALIDATION, report on TEST

In [ ]:
CURRENT_SOURCES = {s["path"] for s in LOADED}

def load_split(split):
    # The Drive files may hold rows from sources removed since (e.g. toxicity); skip those.
    rows = [r for r in read_valid(f"{OUT_DIR}/{split}.jsonl") if r["source"] in CURRENT_SOURCES]
    good = [r for r in rows if r["teacher_letter_mass"] >= MIN_LETTER_MASS]
    return good, len(rows) - len(good)

DATA, DROPPED = {}, {}
for split in ["train", "validation", "test"]:
    DATA[split], DROPPED[split] = load_split(split)
print("dropped (teacher did not answer with a letter):", DROPPED)

USE_SHIFT = True   # yes/no groups also get a nudge toward "yes" (+) or "no" (-)

def probs_at(r, T, b=0.0):
    """Divide the teacher's scores by the dial T. For yes/no rows, also add the
    nudge b to the "true" score. Then turn scores into chances that add up to 1."""
    z = np.array(r["teacher_option_logprobs"]) / T
    if r["type"] == "noul":
        z[1] += b                       # slot 1 is always "true"
    z = z - z.max()
    p = np.exp(z)
    return p / p.sum()

def nll(rows, T, b=0.0):
    """Log loss: a strictly proper scoring rule. Lowest when probabilities match reality."""
    return float(np.mean([-np.log(probs_at(r, T, b)[r["gold_index"]] + 1e-12) for r in rows]))

def ece(rows, T, b=0.0, n_bins=10):
    P    = [probs_at(r, T, b) for r in rows]
    conf = np.array([p.max() for p in P])
    corr = np.array([int(p.argmax() == r["gold_index"]) for p, r in zip(P, rows)])
    total = 0.0
    for k in range(n_bins):
        lo, hi = k / n_bins, (k + 1) / n_bins
        m = (conf >= lo) & ((conf < hi) if k < n_bins - 1 else (conf <= hi))
        if m.any():
            total += m.mean() * abs(corr[m].mean() - conf[m].mean())
    return float(total), float(corr.mean())

GRID   = np.exp(np.linspace(np.log(0.25), np.log(20.0), 200))   # dial settings to try
B_GRID = np.linspace(-8.0, 8.0, 321)                             # nudge sizes to try (step 0.05)

def fit_T(rows):
    return float(GRID[np.argmin([nll(rows, t) for t in GRID])])

def fit_T_and_shift(rows):
    """Yes/no only: try every (dial, nudge) pair and keep the one with the lowest penalty.
    With two options, chance of "true" = sigmoid(gap / T + b), gap = score_true - score_false."""
    gap = np.array([r["teacher_option_logprobs"][1] - r["teacher_option_logprobs"][0] for r in rows])
    y   = np.array([r["gold_index"] for r in rows])                          # 1 = true, 0 = false
    z   = gap[None, None, :] / GRID[:, None, None] + B_GRID[None, :, None]   # (dials, nudges, rows)
    p   = 1.0 / (1.0 + np.exp(-z))
    pen = -(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12)).mean(axis=2)
    i, j = np.unravel_index(np.argmin(pen), pen.shape)
    return float(GRID[i]), float(B_GRID[j])

def fit_group(rows, qtype):
    if qtype == "noul" and USE_SHIFT:
        return fit_T_and_shift(rows)
    return fit_T(rows), 0.0

# Fallback: one (dial, nudge) per question type, fit on all sources together.
# Used when a single (source, type) group has too few validation rows to trust.
POOLED = {}
for qtype in ["choice", "noul"]:
    val = [r for r in DATA["validation"] if r["type"] == qtype]
    POOLED[qtype] = fit_group(val, qtype) if len(val) >= MIN_FIT_ROWS else (1.0, 0.0)

TEMPS, REPORT = {}, []
for g in sorted({(r["source"], r["type"]) for r in DATA["validation"]}):
    val  = [r for r in DATA["validation"] if (r["source"], r["type"]) == g]
    test = [r for r in DATA["test"]       if (r["source"], r["type"]) == g]
    if len(val) >= MIN_FIT_ROWS:
        (T, b), how = fit_group(val, g[1]), "own"
    else:
        (T, b) = POOLED[g[1]]
        how = "pooled" if POOLED[g[1]] != (1.0, 0.0) else "none"
    if how == "own" and T in (float(GRID[0]), float(GRID[-1])):
        print(f"WARNING {g}: best T sits at the edge of the search range ({T:.2f}). "
              f"Look at this group's rows before trusting it.")
    if how == "own" and b in (float(B_GRID[0]), float(B_GRID[-1])):
        print(f"WARNING {g}: best nudge sits at the edge of the search range ({b:+.2f}). "
              f"Look at this group's rows before trusting it.")
    TEMPS[g] = (T, b)
    if not test:
        continue
    T_dial = fit_T(val) if how == "own" else T          # dial alone, for comparison
    e_raw,  acc_raw = ece(test, 1.0)
    e_dial, _       = ece(test, T_dial)
    e_cal,  acc_cal = ece(test, T, b)
    REPORT.append(dict(source=g[0], type=g[1], n_val=len(val), n_test=len(test),
                       acc_raw=acc_raw, acc_cal=acc_cal, T=T, b=b, fit=how,
                       ece_raw=e_raw, ece_dial=e_dial, ece_cal=e_cal,
                       nll_raw=nll(test, 1.0), nll_cal=nll(test, T, b)))

if SMOKE:
    print("SMOKE run: too few rows to fit real temperatures, so T stays 1.0. That's expected.")

import pandas as pd

report_df = pd.DataFrame([{
    "source":    x["source"].split("/")[-1],
    "type":      x["type"],
    "val":       x["n_val"],
    "test":      x["n_test"],
    "acc raw":   f"{x['acc_raw']:.1%}",
    "acc cal":   f"{x['acc_cal']:.1%}",
    "T":         f"{x['T']:.2f}",
    "nudge":     f"{x['b']:+.2f}",
    "fit":       x["fit"],
    "ECE raw":   f"{x['ece_raw']:.3f}",
    "dial only": f"{x['ece_dial']:.3f}",
    "ECE cal":   f"{x['ece_cal']:.3f}",
} for x in REPORT])
print("\nCalibration on the test split")
print(report_df.to_string(index=False))

lean_rows = []
for src in sorted({r["source"] for r in DATA["test"]}):
    rows = [r for r in DATA["test"] if r["source"] == src and r["type"] == "noul"]
    if not rows:
        continue
    T, b = TEMPS.get((src, "noul"), POOLED["noul"])
    lean_rows.append({
        "source":          src.split("/")[-1],
        "said yes before": f"{np.mean([probs_at(r, 1.0).argmax() == 1 for r in rows]):.0%}",
        "said yes after":  f"{np.mean([probs_at(r, T, b).argmax() == 1 for r in rows]):.0%}",
        "actually yes":    f"{np.mean([r['gold_index'] == 1 for r in rows]):.0%}",
    })
print("\nYes/no lean check on test ('after' should be close to 'actually')")
print(pd.DataFrame(lean_rows).to_string(index=False))

for split in DATA:
    for r in DATA[split]:
        T, b = TEMPS.get((r["source"], r["type"]), POOLED.get(r["type"], (1.0, 0.0)))
        r["teacher_temperature"] = round(T, 4)
        r["teacher_shift"]       = round(b, 4)
        r["teacher_probs"]       = [round(float(p), 6) for p in probs_at(r, T, b)]

dropped (teacher did not answer with a letter): {'train': 2, 'validation': 0, 'test': 1}

Calibration on the test split
               source   type  val  test acc raw acc cal    T nudge fit ECE raw dial only ECE cal
            clinc_oos choice  210   209   95.7%   95.7% 3.67 +0.00 own   0.046     0.027   0.027
            clinc_oos   noul   90    91   82.4%   90.1% 1.36 +6.95 own   0.183     0.129   0.067
          go_emotions choice  210   216   60.2%   60.2% 8.29 +0.00 own   0.360     0.064   0.064
          go_emotions   noul   90    84   73.8%   81.0% 9.89 +0.75 own   0.245     0.110   0.119
amazon_massive_intent choice  216   214   92.1%   92.1% 4.57 +0.00 own   0.077     0.024   0.024
amazon_massive_intent   noul   84    86   81.4%   89.5% 3.51 +2.15 own   0.180     0.077   0.091
      amazon_polarity choice  201   216   95.4%   95.4% 3.92 +0.00 own   0.045     0.024   0.024
      amazon_polarity   noul   99    84   95.2%   95.2% 3.51 +0.00 own   0.041     0.055   0.055
       

## [9] LOOK AT RAW ROWS before publishing

In [ ]:
for r in random.Random(1).sample(DATA["test"], min(3, len(DATA["test"]))):
    show = {k: r[k] for k in ["source", "type", "instructions", "state", "options", "gold",
                              "teacher_probs", "teacher_temperature", "teacher_letter_mass"]}
    print(json.dumps(show, indent=2, ensure_ascii=False)[:1500])
    print("-" * 60)

{
  "source": "clinc/clinc_oos",
  "type": "noul",
  "instructions": "Is this request asking for: flight status?",
  "state": "has flight dl123 landed",
  "options": [
    "false",
    "true"
  ],
  "gold": "true",
  "teacher_probs": [
    0.000383,
    0.999617
  ],
  "teacher_temperature": 1.3624,
  "teacher_letter_mass": 0.99976
}
------------------------------------------------------------
{
  "source": "mteb/amazon_massive_intent",
  "type": "choice",
  "instructions": "Which intent best describes this request to a voice assistant?",
  "state": "show latest email",
  "options": [
    "qa currency",
    "cooking query",
    "email query"
  ],
  "gold": "email query",
  "teacher_probs": [
    0.004204,
    0.00166,
    0.994135
  ],
  "teacher_temperature": 4.5739,
  "teacher_letter_mass": 1.0
}
------------------------------------------------------------
{
  "source": "clinc/clinc_oos",
  "type": "choice",
  "instructions": "Which intent best describes this request to a virtual ass

## [10] Publish: PRIVATE first, review on the Hub, then flip to public

In [ ]:
from datasets import Dataset, DatasetDict
from huggingface_hub import DatasetCard

KEEP = ["id", "source", "source_license", "type", "instructions", "state", "options",
        "gold", "gold_index", "display_perm",
        "teacher_option_logprobs", "teacher_temperature", "teacher_shift",
        "teacher_probs", "teacher_letter_mass"]

dd = DatasetDict({s: Dataset.from_list([{k: r[k] for k in KEEP} for r in rows])
                  for s, rows in DATA.items()})
print(dd)

table = "\n".join(
    f"| `{x['source']}` | {x['type']} | {x['n_test']} | {x['acc_raw']:.3f} | {x['acc_cal']:.3f} | "
    f"{x['T']:.2f} | {x['b']:+.2f} | {x['fit']} | {x['ece_raw']:.3f} | {x['ece_dial']:.3f} | "
    f"{x['ece_cal']:.3f} |" for x in REPORT)
source_list = "\n".join(f"- {s.get('credit', s['path'])}" for s in LOADED)

CARD = f"""
# calibrated-typed-decisions (v1)

Typed decisions (multiple choice and yes/no) over short English texts, each with a human
gold label **and** a calibrated soft label from a teacher LLM. Built for training and
evaluating small decision models whose confidence scores mean something.

## Why

A model that says 0.9 should be right about 90% of the time. Instruction-tuned LLMs read
as classifiers tend to be overconfident. This dataset pairs every decision with a teacher
probability distribution that has been calibrated against real answers, plus the raw
ingredients so you can redo the calibration your own way.

## Splits

| split | rows |
|---|---|
| train | {len(DATA['train'])} |
| validation | {len(DATA['validation'])} |
| test | {len(DATA['test'])} |

## Fields

| field | meaning |
|---|---|
| `type` | `choice` or `noul` (yes/no; options are always `["false", "true"]`) |
| `instructions` | the question |
| `state` | the text being judged |
| `options` | option labels in canonical order |
| `gold`, `gold_index` | the human label from the source dataset |
| `teacher_option_logprobs` | teacher log-probability of each option's answer letter, canonical order |
| `teacher_temperature` | temperature fit for this row's (source, type) group |
| `teacher_shift` | yes/no rows only: offset added to the `true` score after dividing by the temperature (0 for choice rows) |
| `teacher_probs` | `softmax(teacher_option_logprobs / teacher_temperature + [0, teacher_shift])` for yes/no, without the shift for choice |
| `teacher_letter_mass` | total probability the teacher put on answer letters at all |
| `display_perm` | the option order the teacher saw (display position to canonical index) |

## How the labels were made

1. Examples come from public, permissively licensed classification datasets (listed below).
2. Each example becomes a question with 3 to 8 shuffled options (the gold label plus random
   distractors), or a yes/no question about one candidate label (about 30% of rows).
3. The teacher (`{TEACHER}`, 4-bit NF4) reads each question once. The answer-letter
   log-probabilities at the first answer position are recorded. Nothing is generated.
4. Rows where the teacher put under {MIN_LETTER_MASS:.0%} of its probability on answer letters are dropped
   (train {DROPPED['train']}, validation {DROPPED['validation']}, test {DROPPED['test']}).
5. One temperature per (source, type) is fit on the validation split by minimizing log loss
   against gold labels, then applied to every row. Yes/no groups also fit a shift toward
   `true` or `false` (a two-parameter Platt-style fit), because the teacher leaned toward
   answering "no": a temperature alone cannot correct a lean. Groups with fewer than
   {MIN_FIT_ROWS} validation rows borrow a pooled fit from all sources of the same type
   (`fit` = `pooled`). Calibration is reported on the test split.

## Teacher calibration on the test split

Accuracy can change after calibration for yes/no rows, because the shift can flip a
borderline answer. Temperature alone never changes the top pick. "ECE dial only" shows
the temperature without the shift, for comparison.

| source | type | n | acc raw | acc calibrated | T | shift | fit | ECE raw | ECE dial only | ECE calibrated |
|---|---|---|---|---|---|---|---|---|---|---|
{table}

## Sources

{source_list}

## Limitations

- Soft labels reflect one teacher's judgment and biases, not human disagreement.
- Calibration is fit per group on average; individual rows can still be off.
- Distractors are sampled at random, so many choice questions are easier than hard-negative ones.
- Options are humanized source label names without descriptions.
- English only. Gold labels inherit any noise in the source datasets.
- A toxicity source (Civil Comments) was scored and left out: the teacher agreed with its
  crowd labels at roughly chance level, so its soft labels carried no reliable signal.
- Yes/no groups: the shift improved accuracy and corrected the teacher's lean toward "no",
  but in some groups the temperature alone scored slightly better on ECE. Both are reported.
- The teacher ran in 4-bit; full-precision logits would differ slightly.

## Usage

```python
from datasets import load_dataset
ds = load_dataset("{HF_REPO}")
row = ds["train"][0]
print(row["options"], row["teacher_probs"], row["gold"])
```
"""

if SMOKE:
    print("SMOKE run: not publishing. Set SMOKE = False and rerun from cell [3].")
else:
    dd.push_to_hub(HF_REPO, private=True)
    card = DatasetCard.load(HF_REPO)
    card.data.license = DATASET_LICENSE
    card.data.language = ["en"]
    card.data.task_categories = ["text-classification"]
    card.data.tags = ["calibration", "soft-labels", "distillation", "typed-decisions"]
    card.data.pretty_name = "Calibrated Typed Decisions"
    card.text = CARD
    card.push_to_hub(HF_REPO)
    print(f"Pushed (private): https://huggingface.co/datasets/{HF_REPO}")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'source_license', 'type', 'instructions', 'state', 'options', 'gold', 'gold_index', 'display_perm', 'teacher_option_logprobs', 'teacher_temperature', 'teacher_shift', 'teacher_probs', 'teacher_letter_mass'],
        num_rows: 4998
    })
    validation: Dataset({
        features: ['id', 'source', 'source_license', 'type', 'instructions', 'state', 'options', 'gold', 'gold_index', 'display_perm', 'teacher_option_logprobs', 'teacher_temperature', 'teacher_shift', 'teacher_probs', 'teacher_letter_mass'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['id', 'source', 'source_license', 'type', 'instructions', 'state', 'options', 'gold', 'gold_index', 'display_perm', 'teacher_option_logprobs', 'teacher_temperature', 'teacher_shift', 'teacher_probs', 'teacher_letter_mass'],
        num_rows: 1499
    })
})


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmpnx5nn0wd.parquet    : 100%|##########|  403kB /  403kB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.65k [00:00<?, ?B/s]

Pushed (private): https://huggingface.co/datasets/m-lagnajit/calibrated-typed-decisions


## [11] Remove Old Smoke Data from the Drive

In [ ]:
import shutil
shutil.rmtree(OUT_DIR, ignore_errors=True)   # deletes ctd_smoke only
os.makedirs(OUT_DIR, exist_ok=True)

## [12] Check Source Dataset Metadata

In [ ]:
from datasets import load_dataset, get_dataset_config_names, get_dataset_split_names

CANDIDATES = ["mteb/banking77", "mteb/amazon_massive_intent", "mteb/toxic_conversations_50k",
              "mteb/amazon_polarity", "mteb/amazon_counterfactual", "m-lagnajit/calibrated-typed-decisions"]

for path in CANDIDATES:
    print("=" * 70)
    print(path, "| license on this copy:", hub_license(path))
    try:
        configs = get_dataset_config_names(path)
        config = "en" if "en" in configs else None
        print("  versions:", configs[:8], "| 'en' available:", "en" in configs, "-> using", config)
        print("  splits:  ", get_dataset_split_names(path, config))
        ds = load_dataset(path, config, split="test")
        print("  test rows:", len(ds))
        print("  columns:  ", ds.features)
        print("  example:  ", {k: str(v)[:80] for k, v in ds[0].items()})
    except Exception as e:
        print("  FAILED:", type(e).__name__, str(e)[:200])

mteb/banking77 | license on this copy: mit
  versions: ['default'] | 'en' available: False -> using None
  splits:   ['train', 'test']
  test rows: 3076
  columns:   {'text': Value('string'), 'label': Value('int64'), 'label_text': Value('string')}
  example:   {'text': 'How do I locate my card?', 'label': '11', 'label_text': 'card_arrival'}
mteb/amazon_massive_intent | license on this copy: apache-2.0


Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

  versions: ['default', 'ta', 'is', 'pl', 'zh-CN', 'el', 'ru', 'te'] | 'en' available: True -> using en


...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4998 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1499 [00:00<?, ? examples/s]

  test rows: 1499
  columns:   {'id': Value('string'), 'source': Value('string'), 'source_license': Value('string'), 'type': Value('string'), 'instructions': Value('string'), 'state': Value('string'), 'options': List(Value('string')), 'gold': Value('string'), 'gold_index': Value('int64'), 'display_perm': List(Value('int64')), 'teacher_option_logprobs': List(Value('float64')), 'teacher_temperature': Value('float64'), 'teacher_shift': Value('float64'), 'teacher_probs': List(Value('float64')), 'teacher_letter_mass': Value('float64')}
  example:   {'id': 'clinc_oos-test-001330', 'source': 'clinc/clinc_oos', 'source_license': 'cc-by-3.0', 'type': 'noul', 'instructions': 'Is this request asking for: directions?', 'state': 'hows the city mpg of this car', 'options': "['false', 'true']", 'gold': 'false', 'gold_index': '0', 'display_perm': '[1, 0]', 'teacher_option_logprobs': '[-6e-05, -16.75006]', 'teacher_temperature': '1.3624', 'teacher_shift': '6.95', 'teacher_probs': '[0.995247, 0.004753]'